# Dataset exploration: Deutsche Bahn monthly data

Source: https://huggingface.co/datasets/piebro/deutsche-bahn-data (CC BY 4.0). One row is one stop event: one train, at one station, on one run.

Change `MONTH` below to look at August (`2026-08`) or September (`2026-09`). Run the cells from top to bottom, because later cells use columns added by earlier ones. A month is about 14.5M rows and needs a few GB of memory.

Sections:
1. Load and first look
2. Categorical columns
3. How `delay_in_min` is defined (the evidence)
4. Cancellations
5. Runs and the `id` column
6. Stations
7. Delay distribution

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 45)

MONTH = "2026-09"  # or "2026-08"
DATA = f"../data/monthly_processed_data/data-{MONTH}.parquet"

raw = pd.read_parquet(DATA)
delay = raw["delay_in_min"]  # the column we want to understand

## 1. Load and first look

In [ ]:
print(f"{MONTH}: {len(raw):,} rows")
raw.dtypes

In [ ]:
# One ICE run, in stop order, to see what a run looks like.
ride = raw.loc[raw["train_type"] == "ICE", "id"].iloc[0]
run_prefix = ride.rsplit("-", 1)[0]  # the id without its last part, the stop number
one_run = raw[raw["id"].str.startswith(run_prefix + "-")]
one_run[
    [
        "train_type",
        "train_number",
        "station_name",
        "train_line_station_num",
        "arrival_planned_time",
        "arrival_change_time",
        "departure_planned_time",
        "departure_change_time",
        "delay_in_min",
        "time",
    ]
].sort_values("train_line_station_num")

In [ ]:
# Share of nulls per column.
raw.isna().mean().rename("null_share").to_frame()

## 2. Categorical columns

In [ ]:
raw["train_type"].value_counts().to_frame()

In [ ]:
# Boolean flags. The three replacement/additional columns are not in the dataset card,
# so their meaning here is a guess from the names.
flags = raw[
    [
        "arrival_is_canceled",
        "departure_is_canceled",
        "is_additional_stop",
        "is_replacement_train",
    ]
].mean()
flags["has_replaced_train_number"] = raw["replaced_train_number"].notna().mean()
flags.rename("share_of_rows").to_frame()

## 3. How `delay_in_min` is defined

**What the dataset card says:** `delay_in_min` is "Delay in minutes" and `time` is "Actual arrival or departure time". It does not say which event a row's delay refers to.

**Question:** each row has both an arrival and a departure (planned and actual). Which one is `delay_in_min`?

**Method:** recompute the arrival delay and the departure delay ourselves from the four time columns, then check which one `delay_in_min` equals. Do it separately for the first stop (no arrival), middle stops (both) and the last stop (no departure), because the answer could differ by position.

A note on missing values: comparing a missing value (NaN) with anything gives False in pandas, so a missing arrival delay never counts as equal to `delay_in_min`.

In [ ]:
def minutes(actual: str, planned: str) -> pd.Series:
    """Minutes between two time columns (NaN where either time is missing)."""
    return (raw[actual] - raw[planned]).dt.total_seconds() / 60


raw["arr_delay"] = minutes("arrival_change_time", "arrival_planned_time")
raw["dep_delay"] = minutes("departure_change_time", "departure_planned_time")

no_arrival = raw["arrival_planned_time"].isna()  # the train starts here
no_departure = raw["departure_planned_time"].isna()  # the train ends here
raw["position"] = np.select(
    [no_arrival & no_departure, no_arrival, no_departure],
    ["none", "first", "last"],
    default="middle",
)
raw["position"].value_counts().to_frame()

### Step 1: eyeball a few middle stops

Look at rows where our recomputed arrival and departure delays differ, and see which one matches `delay_in_min`.

In [ ]:
differs = (
    (raw["position"] == "middle")
    & (raw["arr_delay"] != raw["dep_delay"])
    & ~raw["arrival_is_canceled"]
)
raw.loc[
    differs, ["train_type", "station_name", "arr_delay", "dep_delay", "delay_in_min"]
].head(10)

### Step 2: count it, per position

For each position, the share of rows where `delay_in_min` equals our departure delay and where it equals our arrival delay.

In [ ]:
raw["equals_dep_delay"] = delay == raw["dep_delay"]
raw["equals_arr_delay"] = delay == raw["arr_delay"]

raw.groupby("position").agg(
    rows=("position", "size"),
    share_equals_dep_delay=("equals_dep_delay", "mean"),
    share_equals_arr_delay=("equals_arr_delay", "mean"),
).sort_values("rows", ascending=False)

How to read it:
- First stops: only a departure exists, so `delay_in_min` can only be the departure delay.
- Last stops: only an arrival exists, so it can only be the arrival delay.
- Middle stops are the real test. If `share_equals_dep_delay` is about 1.0 and `share_equals_arr_delay` is clearly lower, then `delay_in_min` is the **departure** delay at middle stops.

### Step 3: does it matter?

If arrival and departure delay were always equal, the question would be pointless. Check how often they differ at middle stops, and by how much.

In [ ]:
middle = raw[(raw["position"] == "middle") & ~raw["arrival_is_canceled"]]
# minutes gained (negative) or lost (positive) while the train stands at the stop
gap = (middle["dep_delay"] - middle["arr_delay"]).dropna()

pd.Series(
    {
        "middle_stops": len(gap),
        "share_arr_ne_dep": (gap != 0).mean(),
        "mean_gap": gap.mean(),
        "p05_gap": gap.quantile(0.05),
        "p95_gap": gap.quantile(0.95),
    }
).to_frame("value")

### Step 4: what is `time`?

The card says "Actual arrival or departure time". Test the guess that `time` is the actual departure time when a departure exists, else the actual arrival time.

In [ ]:
# fillna uses the arrival time wherever the departure time is missing
dep_else_arr = raw["departure_change_time"].fillna(raw["arrival_change_time"])
raw["time_is_dep_else_arr"] = raw["time"] == dep_else_arr
raw["time_is_arr"] = raw["time"] == raw["arrival_change_time"]

raw.groupby("position").agg(
    rows=("position", "size"),
    share_time_is_dep_else_arr=("time_is_dep_else_arr", "mean"),
    share_time_is_arr=("time_is_arr", "mean"),
).sort_values("rows", ascending=False)

### Conclusion

Read the tables above and compare with what the project notes claim:
- `delay_in_min` follows the event of the row: departure for first and middle stops, arrival for the last stop.
- At middle stops this is **not** the arrival delay, and the two differ often enough to matter for transfer planning.
- `time` is the actual time of that same event, so it contains the answer and cannot be a feature.

This is an observation on these files, not documented behaviour. The check can be rerun on any month by changing `MONTH` at the top. For a model target we therefore recompute `arr_delay` and `dep_delay` from the time columns instead of using `delay_in_min`.

## 4. Cancellations

What does `delay_in_min` hold for a canceled stop?

In [ ]:
raw["delay_is_zero"] = delay == 0
raw.groupby("arrival_is_canceled").agg(
    rows=("delay_in_min", "size"),
    mean_delay=("delay_in_min", "mean"),
    max_delay=("delay_in_min", "max"),
    share_zero=("delay_is_zero", "mean"),
)

## 5. Runs and the `id` column

`id` looks like `<ride id>-<run start YYMMDDHHMM>-<stop number>`. `train_line_ride_id` alone repeats across days, so a single run (one trip on one day) is the ride id plus the run start.

In [ ]:
# One regex with two groups: group 0 is the run id, group 1 is the start date inside it.
parts = raw["id"].str.extract(r"^(.*-(\d{6})\d{4})-\d+$")
raw["run_id"] = parts[0]
raw["run_day"] = pd.to_datetime(parts[1], format="%y%m%d")

raw[["id", "train_line_ride_id", "run_id"]].nunique().rename("unique_values").to_frame()

In [ ]:
# Runs per start day. Days outside the month appear when a run started the day before.
raw.groupby("run_day").agg(runs=("run_id", "nunique"), stop_events=("run_id", "size"))

In [ ]:
# Share of stop events whose actual time falls on a different calendar day than the run start.
(raw["time"].dt.normalize() != raw["run_day"]).mean()

## 6. Stations

In [ ]:
print(raw[["eva", "station_name"]].nunique())

# Station names that map to more than one EVA number.
per_name = raw.groupby("station_name").agg(
    n_eva=("eva", "nunique"), rows=("eva", "size")
)
per_name[per_name["n_eva"] > 1].sort_values("n_eva", ascending=False).head(15)

In [ ]:
# Rows without a station name.
raw.loc[raw["station_name"].isna(), ["eva", "xml_station_name", "train_type"]].head(10)

## 7. Delay distribution

Departure delay and arrival delay (recomputed in section 3) for stops that were not canceled, clipped to the range shown.

In [ ]:
LOW, HIGH = -5, 30
ok = raw[~raw["arrival_is_canceled"] & ~raw["departure_is_canceled"]]

fig, ax = plt.subplots(figsize=(8, 3.5))
for column, label in [("dep_delay", "departure delay"), ("arr_delay", "arrival delay")]:
    # share of stops at each whole minute, with outliers piled into the end bins
    share = ok[column].clip(LOW, HIGH).value_counts(normalize=True).sort_index()
    ax.plot(share.index, share.values, label=label)
ax.set_xlabel(f"minutes (clipped to [{LOW}, {HIGH}])")
ax.set_ylabel("share of stops")
ax.legend()
plt.show()

ok[["dep_delay", "arr_delay"]].describe(percentiles=[0.5, 0.9])